
Goal: Assess specificity of EA scores.

In [16]:
# libraries

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
import numpy as np
from scipy.stats import spearmanr
from scipy.stats import percentileofscore
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests

Prepare lists.

In [17]:
targets = ['P03_joy', 'P29_joy', 'P34_joy', 'P42_joy', 'P44_joy',
           'P12_sadness', 'P15_sadness', 'P27_sadness', 'P34_sadness', 'P44_sadness', 
           'P03_anger', 'P15_anger', 'P22_anger', 'P38_anger', 'P42_anger',
           'P17_pride', 'P22_pride', 'P29_pride', 'P42_pride', 'P44_pride',
           'P03_shame', 'P22_shame', 'P37_shame', 'P38_shame', 'P45_shame', 
           'P12_neutral', 'P17_neutral', 'P29_neutral', 'P34_neutral', 'P38_neutral']

perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23",
                "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# emotions on the Geneva Emotion Wheel (for specific emotion selection)
geneva_emotions = ['geneva_interest', 'geneva_amusement',	'geneva_pride',	'geneva_joy', 'geneva_pleasure',
                   'geneva_contentment',	'geneva_love',	'geneva_admiration',	'geneva_relief',	'geneva_compassion',
                   'geneva_sadness',	'geneva_guilt',	'geneva_regret',	'geneva_shame',	'geneva_disappointment',
                   'geneva_fear',	'geneva_disgust',	'geneva_contempt',	'geneva_hate',	'geneva_anger']

# data from previous script (EA scores; correspondance between each target and perceiver ratings pair)
correlation_data = pd.read_csv("outputs/correlation_EA_scores.csv") 
geneva_data = pd.read_csv("outputs/geneva_EA_scores.csv")

r-to-z transformation for correlations.

FORMULA: zr = ln((1+r) / (1-r)) / 2

In [ ]:
def r_to_z(r_value):
    return np.log((1 + r_value) / (1 - r_value)) / 2

correlation_data['z_corr'] = r_to_z(correlation_data['spearman_r'])

# plot r-values and z-values
plt.figure(figsize=(7, 5))
plt.hist(correlation_data['spearman_r'], bins=30, color='blue', label='r') # r-values
plt.hist(correlation_data['z_corr'], bins=30, color='red', label='z', alpha=.3) # z-values
plt.xlabel('value')
plt.ylabel('count')
plt.title('raw r-values vs fisher z-values')
plt.legend()
plt.show()


Graph to show the distribution of matching and nonmatching correlations. 

matching: correlation values betweeen perceivers and the target they actually watched\
nonmatching: surrogate correlations between perceivers and targets they did not watch\
I also look at surrogate correlations with targets of the *same emotion category* as the one the perceiver actually watched. 


In [ ]:
def match_category(row):
    if row['match_type'] == 'match':
        return 'match' # matching correlation
    elif row['emotion_type'] == 'same_emotion' and row['match_type'] == 'nonmatch':
        return 'nonmatch_same' # nonmatching (and not same emotion as the actual target)
    elif row['emotion_type'] == 'diff_emotion':
        return 'nonmatch_diff' # nonmatching (and same emotion as the actual target)
    
# put this information in a new column
correlation_data['match_category'] = correlation_data.apply(match_category, axis=1) # axis 1 = apply to rows

# make the graph
nrow = 6 
ncol = 5 
fig, axs = plt.subplots(nrow, ncol, figsize=(13, 15))
axs = axs.flatten()  # numbered list of plots; easier for looping

for i, ax in enumerate(axs):
    target = targets[i]
    data = correlation_data[correlation_data['evaluation_of'] == target]

    ax.hist(data[data['match_category'] == 'nonmatch_diff']['z_corr'],
            bins=30, alpha=0.4, color='navy', label='nonmatch_diff', edgecolor='none')

    ax.hist(data[data['match_category'] == 'nonmatch_same']['z_corr'],
            bins=30, alpha=0.5, color='cyan', label='nonmatch_same', edgecolor='none')

    ax.hist(data[data['match_category'] == 'match']['z_corr'],
            bins=30, alpha=1.0, color='red', label='match', edgecolor='none')

    ax.set_title(target)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_ylim(0,40)
    ax.set_xlim(-3,3)

fig.suptitle('Matching vs. Nonmatching Distribution (Correlation EA scores)', fontsize=16)
fig.legend(['nonmatch_diff', 'nonmatch_same', 'match'], loc='upper right')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


### make the same graph, but for specific emotion recognition (Geneva Emotion Wheel) 

geneva_data['match_category'] = geneva_data.apply(match_category, axis=1) # axis 1 = apply to rows

nrow = 6 
ncol = 5 
fig, axs = plt.subplots(nrow, ncol, figsize=(13, 15))
axs = axs.flatten()  # numbered list of plots; easier for looping

for i, ax in enumerate(axs):
    target = targets[i]
    data = geneva_data[geneva_data['evaluation_of'] == target]

    ax.hist(data[data['match_category'] == 'nonmatch_diff']['sum_distance'],
            bins=30, alpha=0.4, color='navy', label='nonmatch_diff', edgecolor='none')

    ax.hist(data[data['match_category'] == 'nonmatch_same']['sum_distance'],
            bins=30, alpha=0.5, color='cyan', label='nonmatch_same', edgecolor='none')

    ax.hist(data[data['match_category'] == 'match']['sum_distance'],
            bins=30, alpha=1.0, color='red', label='match', edgecolor='none')

    ax.set_title(target)
    ax.axvline(50, color='gray', linestyle='--')
    ax.set_ylim(0,40)
    ax.set_xlim(0,100)

fig.suptitle('Matching vs. Nonmatching Distribution (GEW EA scores)', fontsize=16)
fig.legend(['nonmatch_diff', 'nonmatch_same', 'match'], loc='upper right')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()



We want to test for target-specificity. Step #1: Calculate percentile value of each target-perceiver pair. And add this value in a new column in the EA scores dataset.\
Also check percentile distribution within target

In [20]:
### for correlation EA scores

full_distribution_corr = correlation_data['z_corr']
percentile_in_full_distribution_corr = [] # percentile of a value in the full distribution
percentile_in_target_corr = [] # percentile of a value relative to just the corresponding target

# go through each row and calculate percentile 
for idx, row in correlation_data.iterrows():
    
    z = row['z_corr']
    target = row['evaluation_of']
    
    # percentile in full data
    percentile_corr = percentileofscore(full_distribution_corr, z, kind='mean')
    percentile_in_full_distribution_corr.append(percentile_corr)

    # percentile in target data
    target_dist = correlation_data[correlation_data['evaluation_of'] == target]['z_corr']
    percentile_corr_target = percentileofscore(target_dist, z, kind='mean')
    percentile_in_target_corr.append(percentile_corr_target)
    
# add to current dataframe
correlation_data['percentile_in_full_dist_corr'] = percentile_in_full_distribution_corr
correlation_data['percentile_in_target_dist_corr'] = percentile_in_target_corr

# re-export with added column
correlation_data.to_csv("outputs/correlation_EA_scores.csv", index=False)

############################ same for GEW scores

full_distribution_gew = geneva_data['sum_distance'].values
percentile_in_full_distribution_gew = []
percentile_in_target_gew = [] 

for idx, row in geneva_data.iterrows():
    score = row['sum_distance']
    target = row['evaluation_of']

    # percentile in full dataset
    percentile_gew = percentileofscore(full_distribution_gew, score, kind='mean')
    percentile_in_full_distribution_gew.append(percentile_gew)

    # percentile in target data
    target_dist = geneva_data[geneva_data['evaluation_of'] == target]['sum_distance']
    percentile_gew_target = percentileofscore(target_dist, score, kind='mean')
    percentile_in_target_gew.append(percentile_gew_target)
    
# add to current dataframe
geneva_data['percentile_in_full_dist_gew'] = percentile_in_full_distribution_gew
geneva_data['percentile_in_target_dist_gew'] = percentile_in_target_gew

# re-export
geneva_data.to_csv("outputs/geneva_EA_scores.csv", index=False)

One-sample t-test for each measure, to test if matching scores are significantly higher than the null distribution

In [ ]:

# correlation EA scores
full_distribution_corr = correlation_data['percentile_in_full_dist_corr']
match_distribution_corr = correlation_data[correlation_data['match_type'] == 'match']['percentile_in_full_dist_corr'].dropna()
t_value, p_value = ttest_1samp(match_distribution_corr, popmean=full_distribution_corr.mean())

# print results
dof = len(match_distribution_corr) - 1 # degrees of freedom
print(f"Matching: M = {match_distribution_corr.mean():.2f}, SD = {match_distribution_corr.std():.2f}")
print(f"Full distribution: M = {full_distribution_corr.mean():.2f}, SD = {full_distribution_corr.std():.2f}")
print(f"One-sample t-test: t({dof}) = {t_value:.2f}, p = {p_value:.4f}")

# make it into a graph
groups = ['Match', 'Null']
data = [match_distribution_corr, full_distribution_corr]

plt.figure(figsize=(5, 6))

for i, values in enumerate(data): # loop through values of each group
    
    # 50% box (from percentile 25 to 75)
    plt.fill_between([i - 0.2, i + 0.2], # add thickness
                     np.percentile(values, 25), # starts at percentile 25
                     np.percentile(values, 75), # ends at percentile 75
                     color='lightgray', alpha=1.0, edgecolor='black', linewidth=1)

    # horizontal mean line
    plt.plot([i - 0.2, i + 0.2], 
             [values.mean(), values.mean()], 
             color='black', linewidth=2)

    # sd vertical lines
    plt.plot([i, i], [np.percentile(values, 75), # start outside the box
                      values.mean() + values.std()], color='black', linewidth=1.2)
    plt.plot([i, i], [np.percentile(values, 25), # start outside the box
                      values.mean() - values.std()], color='black', linewidth=1.2)

    # horizontal line at +/- 1 SD
    plt.plot([i - 0.1, i + 0.1], # not as wide as the box
             [values.mean() + values.std(), 
              values.mean() + values.std()], 
             color='black', linewidth=1)
    plt.plot([i - 0.1, i + 0.1], 
             [values.mean() - values.std(), 
              values.mean() - values.std()], 
             color='black', linewidth=1)
    
ax = plt.gca()  # get axes
ax.grid(False) # remove background grid

plt.xticks([0, 1], groups, fontsize=12) # label each group
plt.yticks(np.arange(0, 110, 10), fontsize=10) # ticks at each jump of 10 (max. 110 so we can see '100')
plt.ylabel('Percentile', fontsize=12)
plt.axhline(50, color='gray', linestyle=':', linewidth=2)  # dashed line at 50
plt.title('Percentile Distribution of Matching EA Scores (correlation) vs. Null Distribution', fontsize=13, pad=15)
plt.ylim(0, 100)
plt.show()

############################################## same with GEW scores

full_distribution_gew = geneva_data['percentile_in_full_dist_gew']
match_distribution_gew = geneva_data[geneva_data['match_type'] == 'match']['percentile_in_full_dist_gew'].dropna()
t_value, p_value = ttest_1samp(match_distribution_gew, popmean=full_distribution_gew.mean())

# print results
dof = len(match_distribution_gew) - 1 # degrees of freedom
print(f"Matching: M = {match_distribution_gew.mean():.2f}, SD = {match_distribution_gew.std():.2f}")
print(f"Full distribution: M = {full_distribution_gew.mean():.2f}, SD = {full_distribution_gew.std():.2f}")
print(f"One-sample t-test: t({dof}) = {t_value:.2f}, p = {p_value:.4f}")

# make it into a graph
groups = ['Match', 'Null']
data = [match_distribution_gew, full_distribution_gew]

plt.figure(figsize=(5, 6))

for i, values in enumerate(data): # loop through values of each group
    
    # 50% box (from percentile 25 to 75)
    plt.fill_between([i - 0.2, i + 0.2], # add thickness
                     np.percentile(values, 25), # starts at percentile 25
                     np.percentile(values, 75), # ends at percentile 75
                     color='lightgray', alpha=1.0, edgecolor='black', linewidth=1)

    # horizontal mean line
    plt.plot([i - 0.2, i + 0.2], 
             [values.mean(), values.mean()], 
             color='black', linewidth=2)

    # sd vertical lines
    plt.plot([i, i], [np.percentile(values, 75), # start outside the box
                      values.mean() + values.std()], color='black', linewidth=1.2)
    plt.plot([i, i], [np.percentile(values, 25), # start outside the box
                      values.mean() - values.std()], color='black', linewidth=1.2)

    # horizontal line at +/- 1 SD
    plt.plot([i - 0.1, i + 0.1], # not as wide as the box
             [values.mean() + values.std(), 
              values.mean() + values.std()], 
             color='black', linewidth=1)
    plt.plot([i - 0.1, i + 0.1], 
             [values.mean() - values.std(), 
              values.mean() - values.std()], 
             color='black', linewidth=1)

ax = plt.gca()  # get axes
ax.grid(False) # remove background grid

plt.xticks([0, 1], groups, fontsize=12) # label each group
plt.yticks(np.arange(0, 110, 10), fontsize=10) # ticks at each jump of 10 (110 so we can see '100')
plt.ylabel('Percentile', fontsize=12)
plt.axhline(50, color='gray', linestyle=':', linewidth=2)  # dashed line at 50
plt.title('Percentile Distribution of Matching EA Scores (GEW) vs. Null Distribution', fontsize=13, pad = 15)
plt.ylim(0, 100)
plt.show()

Now test per-target specificity. So get the percentile ditribution per target, and do the same test with correction for multiple comparisons

In [ ]:

target_specificity_results_corr = []

for target in targets:
    
    full_distribution = correlation_data[correlation_data['evaluation_of'] == target]["percentile_in_target_dist_corr"].dropna()
    match_distribution = correlation_data[(correlation_data['evaluation_of'] == target) & (correlation_data['match_type'] == 'match')]["percentile_in_target_dist_corr"].dropna()

    t_value, p_value = ttest_1samp(match_distribution, popmean=full_distribution.mean())
    
    target_specificity_results_corr.append({
        "target": target,
        "mean_matching": match_distribution.mean(),
        "sd_matching": match_distribution.std(),
        "mean_full_dist": full_distribution.mean(),
        "sd_full_dist": full_distribution.std(),
        "n": len(match_distribution),
        "t": t_value,
        "p": p_value})

target_spec_corr = pd.DataFrame(target_specificity_results_corr)

reject, p_corr, _, _ = multipletests(target_spec_corr['p'], method='fdr_bh')
target_spec_corr['p_corrected'] = p_corr
target_spec_corr['significant'] = reject

print(target_spec_corr)


###################### same thing for GEW

target_specificity_results_gew = []

for target in targets:
    
    full_distribution = geneva_data[geneva_data['evaluation_of'] == target]["percentile_in_target_dist_gew"].dropna()
    match_distribution = geneva_data[(geneva_data['evaluation_of'] == target) & (geneva_data['match_type'] == 'match')]["percentile_in_target_dist_gew"].dropna()

    t_value, p_value = ttest_1samp(match_distribution, popmean=full_distribution.mean())
    
    target_specificity_results_gew.append({
        "target": target,
        "mean_matching": match_distribution.mean(),
        "sd_matching": match_distribution.std(),
        "mean_full_dist": full_distribution.mean(),
        "sd_full_dist": full_distribution.std(),
        "n": len(match_distribution),
        "t": t_value,
        "p": p_value})

target_spec_gew = pd.DataFrame(target_specificity_results_gew)

reject, p_corr, _, _ = multipletests(target_spec_gew['p'], method='fdr_bh')
target_spec_gew['p_corrected'] = p_corr
target_spec_gew['significant'] = reject

print(target_spec_gew)

Make graph for target specificity

In [ ]:

# only keep matching scores
match_data = correlation_data[correlation_data['match_type'] == 'match']

fig, ax = plt.subplots(figsize=(15, 8))

for i, target in enumerate(targets):
    
    values = match_data[match_data['evaluation_of'] == target]['percentile_in_target_dist_corr'].dropna()

    # different color depending on p value
    p = target_spec_corr[target_spec_corr['target'] == target]['p_corrected'].values[0]
    color = 'gray' if p < 0.05 else 'red'
    
    # 50% box
    ax.fill_between([i - 0.2, i + 0.2], np.percentile(values, 25), np.percentile(values, 75), color=color, edgecolor='black', linewidth=0.8)
    
    # mean horizontal line
    ax.plot([i - 0.2, i + 0.2], [np.mean(values), np.mean(values)], color='black', linewidth=2)
    
    # SD if outside box (some boxes are longer than SD from mean)
    if np.mean(values) + np.std(values) > np.percentile(values, 75):
        ax.plot([i, i], [np.percentile(values, 75), np.mean(values) + np.std(values)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) + np.std(values), np.mean(values) + np.std(values)], color='black')
    if np.mean(values) - np.std(values) < np.percentile(values, 25):
        ax.plot([i, i], [np.mean(values) - np.std(values), np.percentile(values, 25)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) - np.std(values), np.mean(values) - np.std(values)], color='black')


ax.grid(False) # remove background grid

ax.set_xticks(np.arange(len(targets)))
ax.set_xticklabels(targets, rotation=90)
ax.set_ylim(0, 110) # 110 so we can see '100'
ax.axhline(50, linestyle='--', color='gray')
ax.set_ylabel("Percentile")
ax.set_title("Correlation EA – Match Percentiles per Target")

ax.bar(0, 0, color='gray', label='Significant (p<.05)')
ax.bar(0, 0, color='red', label='Not significant (p>.05)')
ax.legend(loc='upper right')
plt.show()


###### same thing for Geneva Emotion Wheel

# only keep matching scores
match_data = geneva_data[geneva_data['match_type'] == 'match']

fig, ax = plt.subplots(figsize=(15, 8))

for i, target in enumerate(targets):
    
    values = match_data[match_data['evaluation_of'] == target]['percentile_in_target_dist_gew'].dropna()

    # different color depending on p value
    p = target_spec_gew[target_spec_gew['target'] == target]['p_corrected'].values[0]
    color = 'gray' if p < 0.05 else 'red'
    
    # 50% box
    ax.fill_between([i - 0.2, i + 0.2], np.percentile(values, 25), np.percentile(values, 75), color=color, edgecolor='black', linewidth=0.8)
    
    # mean horizontal line
    ax.plot([i - 0.2, i + 0.2], [np.mean(values), np.mean(values)], color='black', linewidth=2)
    
    # SD, only if outside box (some boxes are longer than SD from mean)
    if np.mean(values) + np.std(values) > np.percentile(values, 75):
        ax.plot([i, i], [np.percentile(values, 75), np.mean(values) + np.std(values)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) + np.std(values), np.mean(values) + np.std(values)], color='black')
    if np.mean(values) - np.std(values) < np.percentile(values, 25):
        ax.plot([i, i], [np.mean(values) - np.std(values), np.percentile(values, 25)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) - np.std(values), np.mean(values) - np.std(values)], color='black')


ax.grid(False) # remove background grid

ax.set_xticks(np.arange(len(targets)))
ax.set_xticklabels(targets, rotation=90)
ax.set_ylim(0, 110) # 110 so we can see '100'
ax.axhline(50, linestyle='--', color='gray')
ax.set_ylabel("Percentile")
ax.set_title("GEW EA – Match Percentiles per Target")

ax.bar(0, 0, color='gray', label='Significant (p<.05)')
ax.bar(0, 0, color='red', label='Not significant (p>.05)')
ax.legend(loc='upper right')
plt.show()